# 09 — Temporal / Semi-Dynamic Incremental Benchmark

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

Stage 08 established a strong static benchmark.

The purpose of Stage 09 is different:

> Test whether time-sensitive factors add predictive value beyond the static susceptibility factors.

---

# Core methodological rule

For every temporal experiment, compare:

### A. Static-only model
and

### B. Static + temporal model

using:

- exactly the same case/control rows;
- exactly the same spatial validation groups;
- exactly the same outer folds;
- exactly the same algorithm family.

This prevents a misleading comparison caused by changing the dataset at the same time as changing the predictors.

---

# Temporal experiments

## Experiment 1 — Rainfall

Static predictors

vs

Static +:
- 1-day antecedent rainfall
- 3-day antecedent rainfall
- 7-day antecedent rainfall
- 30-day antecedent rainfall

---

## Experiment 2 — Rainfall + ERA5 soil moisture

Static predictors

vs

Static + rainfall +:
- ERA5 0–7 cm soil moisture
- ERA5 7–28 cm soil moisture

---

## Experiment 3 — Historical land cover

Static predictors

vs

Static + harmonised historical land-cover class

---

## Experiment 4 — Full temporal enhanced

Static predictors

vs

Static +:
- rainfall
- ERA5 soil moisture
- historical land cover
- current waterway distance

Historical road distance is intentionally excluded for now because its historical OSM completeness still needs a separate quality assessment.

---

# Algorithms

The temporal experiment uses all three algorithms:

1. Logistic Regression
2. Random Forest
3. XGBoost

The same spatially aware nested validation methodology is retained.


In [3]:
# Optional:
%pip install xgboost scikit-learn pandas numpy

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    StratifiedGroupKFold,
    RandomizedSearchCV,
    cross_val_predict,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

print("pandas:", pd.__version__)


Note: you may need to restart the kernel to use updated packages.
pandas: 3.0.2



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Project paths

In [4]:
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

TEMPORAL_DIR = (
    DEV_DIR
    / "04_model_ready"
)

OUTPUT_DIR = (
    DEV_DIR
    / "05_model_results"
    / "temporal_incremental"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATASET_PATHS = {
    "rainfall":
        TEMPORAL_DIR
        / "temporal_model_rainfall_spatial.csv",

    "rainfall_soilmoisture":
        TEMPORAL_DIR
        / "temporal_model_rainfall_soilmoisture_spatial.csv",

    "landcover":
        TEMPORAL_DIR
        / "temporal_model_landcover_spatial.csv",

    "full_enhanced":
        TEMPORAL_DIR
        / "temporal_model_full_enhanced_spatial.csv",
}

for name, path in DATASET_PATHS.items():
    print(name, "->", path, "| exists:", path.exists())


rainfall -> C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\temporal_model_rainfall_spatial.csv | exists: True
rainfall_soilmoisture -> C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\temporal_model_rainfall_soilmoisture_spatial.csv | exists: True
landcover -> C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\temporal_model_landcover_spatial.csv | exists: True
full_enhanced -> C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\temporal_model_full_enhanced_spatial.csv | exists: True


## 2. Shared static predictors

These are the same core static factors used before the enhanced context stage.

Using these within each temporal subset gives a clean incremental comparison.


In [5]:
STATIC_NUMERIC = [
    "elevation_m",
    "slope_deg",
    "aspect_sin",
    "aspect_cos",
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
    "distance_major_fault_m",
]

STATIC_CATEGORICAL = [
    c
    for c in [
        "geology_age",
        "geology_unit_type",
    ]
]

print("Static numerical:", STATIC_NUMERIC)
print("Static categorical candidates:", STATIC_CATEGORICAL)


Static numerical: ['elevation_m', 'slope_deg', 'aspect_sin', 'aspect_cos', 'soil_clay', 'soil_sand', 'soil_silt', 'soil_bulk_density', 'distance_major_fault_m']
Static categorical candidates: ['geology_age', 'geology_unit_type']


# PART A — TEMPORAL DATASET DEFINITIONS

## 3. Feature sets for each experiment


In [6]:
EXPERIMENTS = {
    "rainfall": {
        "temporal_numeric": [
            "rain_1d_mm",
            "rain_3d_mm",
            "rain_7d_mm",
            "rain_30d_mm",
        ],
        "temporal_categorical": [],
    },

    "rainfall_soilmoisture": {
        "temporal_numeric": [
            "rain_1d_mm",
            "rain_3d_mm",
            "rain_7d_mm",
            "rain_30d_mm",
            "soil_moisture_0_7cm_7d_mean",
            "soil_moisture_7_28cm_7d_mean",
        ],
        "temporal_categorical": [],
    },

    "landcover": {
        "temporal_numeric": [],
        "temporal_categorical": [
            "landcover_harmonised",
        ],
    },

    "full_enhanced": {
        "temporal_numeric": [
            "rain_1d_mm",
            "rain_3d_mm",
            "rain_7d_mm",
            "rain_30d_mm",
            "soil_moisture_0_7cm_7d_mean",
            "soil_moisture_7_28cm_7d_mean",
            "distance_waterway_current_m",
        ],
        "temporal_categorical": [
            "landcover_harmonised",
        ],
    },
}


## 4. Load datasets and define the binary target

- `case` = 1
- `matched_control` = 0


In [7]:
datasets = {}

for name, path in DATASET_PATHS.items():

    if not path.exists():
        print("SKIP missing:", name)
        continue

    temp = pd.read_csv(path)

    temp["target"] = (
        temp["temporal_role"]
        .eq("case")
        .astype(int)
    )

    datasets[name] = temp

    print(
        name,
        "| rows:",
        len(temp),
        "| cases:",
        int(temp["target"].sum()),
        "| controls:",
        int((temp["target"] == 0).sum()),
        "| validation groups:",
        temp["validation_group"].nunique(),
    )


rainfall | rows: 567 | cases: 189 | controls: 378 | validation groups: 38
rainfall_soilmoisture | rows: 474 | cases: 158 | controls: 316 | validation groups: 34
landcover | rows: 444 | cases: 148 | controls: 296 | validation groups: 33
full_enhanced | rows: 363 | cases: 121 | controls: 242 | validation groups: 28


# PART B — LEAKAGE-SAFE PIPELINE

## 5. Preprocessing builder

All imputation, scaling and categorical encoding occur inside the CV pipeline.


In [8]:
def build_preprocessor(
    numeric_features,
    categorical_features,
    scale_numeric
):

    numeric_steps = [
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]

    if scale_numeric:
        numeric_steps.append(
            (
                "scaler",
                StandardScaler()
            )
        )

    numeric_pipe = Pipeline(
        numeric_steps
    )

    transformers = [
        (
            "num",
            numeric_pipe,
            numeric_features
        )
    ]

    if categorical_features:

        categorical_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            ),
        ])

        transformers.append(
            (
                "cat",
                categorical_pipe,
                categorical_features
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


## 6. Model/search builder

Search spaces are regularized rather than aggressively complex.


In [9]:
RANDOM_SEED = 42

def build_model_specs(
    numeric_features,
    categorical_features
):

    lr_pre = build_preprocessor(
        numeric_features,
        categorical_features,
        scale_numeric=True
    )

    tree_pre = build_preprocessor(
        numeric_features,
        categorical_features,
        scale_numeric=False
    )

    lr = Pipeline([
        ("preprocess", lr_pre),
        (
            "model",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_SEED
            )
        ),
    ])

    rf = Pipeline([
        ("preprocess", tree_pre),
        (
            "model",
            RandomForestClassifier(
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=-1
            )
        ),
    ])

    xgb = Pipeline([
        ("preprocess", tree_pre),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_SEED,
                n_jobs=-1,
                tree_method="hist"
            )
        ),
    ])

    return {
        "Logistic Regression": {
            "pipeline": lr,
            "params": {
                "model__C": [
                    0.01,
                    0.03,
                    0.1,
                    0.3,
                    1.0,
                    3.0,
                    10.0,
                ],
            },
            "n_iter": 7,
        },

        "Random Forest": {
            "pipeline": rf,
            "params": {
                "model__n_estimators": [
                    300,
                    500,
                    800,
                ],
                "model__max_depth": [
                    4,
                    6,
                    8,
                    12,
                    None,
                ],
                "model__min_samples_leaf": [
                    2,
                    4,
                    6,
                    10,
                ],
                "model__min_samples_split": [
                    5,
                    10,
                    20,
                ],
                "model__max_features": [
                    "sqrt",
                    "log2",
                    0.5,
                ],
            },
            "n_iter": 20,
        },

        "XGBoost": {
            "pipeline": xgb,
            "params": {
                "model__n_estimators": [
                    100,
                    200,
                    350,
                    500,
                ],
                "model__max_depth": [
                    2,
                    3,
                    4,
                ],
                "model__learning_rate": [
                    0.01,
                    0.03,
                    0.05,
                    0.1,
                ],
                "model__subsample": [
                    0.65,
                    0.8,
                    1.0,
                ],
                "model__colsample_bytree": [
                    0.65,
                    0.8,
                    1.0,
                ],
                "model__min_child_weight": [
                    3,
                    5,
                    8,
                ],
                "model__reg_alpha": [
                    0.0,
                    0.1,
                    0.5,
                ],
                "model__reg_lambda": [
                    2.0,
                    5.0,
                    10.0,
                ],
                "model__scale_pos_weight": [
                    1.0,
                    1.5,
                    2.0,
                ],
            },
            "n_iter": 24,
        },
    }


# PART C — THRESHOLD AND METRICS

## 7. Metric helpers


In [10]:
def calculate_metrics(
    y_true,
    probability,
    threshold
):

    prediction = (
        np.asarray(probability)
        >= threshold
    ).astype(int)

    return {
        "accuracy":
            accuracy_score(
                y_true,
                prediction
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                prediction
            ),

        "precision":
            precision_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                probability
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                probability
            ),
    }


def select_threshold(
    y_true,
    probability
):

    candidates = np.linspace(
        0.20,
        0.80,
        121
    )

    rows = []

    for t in candidates:

        m = calculate_metrics(
            y_true,
            probability,
            t
        )

        rows.append({
            "threshold": t,
            "balanced_accuracy":
                m[
                    "balanced_accuracy"
                ],
            "recall":
                m[
                    "recall"
                ],
            "f1":
                m[
                    "f1"
                ],
        })

    table = pd.DataFrame(
        rows
    )

    best = table.sort_values(
        [
            "balanced_accuracy",
            "recall",
            "f1"
        ],
        ascending=False
    ).iloc[0]

    return float(
        best[
            "threshold"
        ]
    )


# PART D — ONE FAIR PAIRED BENCHMARK

## 8. Benchmark helper

For one dataset and one algorithm:

- create shared 5-fold spatial outer splits;
- tune model hyperparameters inside each outer training fold;
- choose threshold from inner-training OOF predictions;
- evaluate on untouched outer test fold.

This function is called twice using the same folds:

1. static-only predictors
2. static + temporal predictors


In [11]:
METRIC_NAMES = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]


def run_variant(
    dataset_name,
    variant_name,
    data,
    numeric_features,
    categorical_features,
    outer_folds
):

    X = data[
        numeric_features
        + categorical_features
    ].copy()

    y = data[
        "target"
    ].astype(int).copy()

    groups = data[
        "validation_group"
    ].astype(str).copy()

    specs = build_model_specs(
        numeric_features,
        categorical_features
    )

    result_rows = []
    oof_rows = []

    for model_name, spec in specs.items():

        print(
            "\n",
            dataset_name,
            "|",
            variant_name,
            "|",
            model_name
        )

        for fold, (train_idx, test_idx) in enumerate(
            outer_folds,
            start=1
        ):

            X_train = X.iloc[
                train_idx
            ].copy()

            y_train = y.iloc[
                train_idx
            ].copy()

            g_train = groups.iloc[
                train_idx
            ].copy()

            X_test = X.iloc[
                test_idx
            ].copy()

            y_test = y.iloc[
                test_idx
            ].copy()

            inner = StratifiedGroupKFold(
                n_splits=3,
                shuffle=True,
                random_state=(
                    RANDOM_SEED
                    + fold
                )
            )

            inner_splits = list(
                inner.split(
                    X_train,
                    y_train,
                    g_train
                )
            )

            search = RandomizedSearchCV(
                estimator=clone(
                    spec[
                        "pipeline"
                    ]
                ),
                param_distributions=
                    spec[
                        "params"
                    ],
                n_iter=
                    spec[
                        "n_iter"
                    ],
                scoring="roc_auc",
                cv=inner_splits,
                refit=True,
                random_state=(
                    RANDOM_SEED
                    + fold
                ),
                n_jobs=-1,
                verbose=0,
            )

            search.fit(
                X_train,
                y_train
            )

            best = search.best_estimator_

            inner_oof = cross_val_predict(
                clone(
                    best
                ),
                X_train,
                y_train,
                cv=inner_splits,
                method="predict_proba",
                n_jobs=-1
            )[:, 1]

            threshold = select_threshold(
                y_train,
                inner_oof
            )

            best.fit(
                X_train,
                y_train
            )

            train_prob = best.predict_proba(
                X_train
            )[:, 1]

            test_prob = best.predict_proba(
                X_test
            )[:, 1]

            train_metrics = calculate_metrics(
                y_train,
                train_prob,
                threshold
            )

            test_metrics = calculate_metrics(
                y_test,
                test_prob,
                threshold
            )

            row = {
                "dataset":
                    dataset_name,

                "variant":
                    variant_name,

                "model":
                    model_name,

                "fold":
                    fold,

                "threshold":
                    threshold,
            }

            for metric in METRIC_NAMES:

                row[
                    f"train_{metric}"
                ] = train_metrics[
                    metric
                ]

                row[
                    f"test_{metric}"
                ] = test_metrics[
                    metric
                ]

                row[
                    f"gap_{metric}"
                ] = (
                    train_metrics[
                        metric
                    ]
                    -
                    test_metrics[
                        metric
                    ]
                )

            result_rows.append(
                row
            )

            for pos, original_idx in enumerate(
                test_idx
            ):

                oof_rows.append({
                    "dataset":
                        dataset_name,

                    "variant":
                        variant_name,

                    "model":
                        model_name,

                    "fold":
                        fold,

                    "row_index":
                        int(
                            original_idx
                        ),

                    "target":
                        int(
                            y.iloc[
                                original_idx
                            ]
                        ),

                    "probability":
                        float(
                            test_prob[
                                pos
                            ]
                        ),

                    "threshold":
                        threshold,
                })

    return (
        pd.DataFrame(
            result_rows
        ),
        pd.DataFrame(
            oof_rows
        )
    )


# PART E — RUN ALL TEMPORAL EXPERIMENTS

## 9. Run paired static vs temporal comparisons


In [12]:
all_results = []
all_oof = []

for dataset_name, data in datasets.items():

    config = EXPERIMENTS[
        dataset_name
    ]

    # Use only categorical static fields that truly exist.
    static_categorical = [
        c
        for c in STATIC_CATEGORICAL
        if c in data.columns
        and data[c].notna().any()
    ]

    static_numeric = [
        c
        for c in STATIC_NUMERIC
        if c in data.columns
    ]

    temporal_numeric = [
        c
        for c in config[
            "temporal_numeric"
        ]
        if c in data.columns
    ]

    temporal_categorical = [
        c
        for c in config[
            "temporal_categorical"
        ]
        if c in data.columns
    ]

    y = data[
        "target"
    ].astype(int)

    groups = data[
        "validation_group"
    ].astype(str)

    outer = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_SEED
    )

    outer_folds = list(
        outer.split(
            data,
            y,
            groups
        )
    )

    static_results, static_oof = run_variant(
        dataset_name,
        "Static only",
        data,
        static_numeric,
        static_categorical,
        outer_folds
    )

    enhanced_results, enhanced_oof = run_variant(
        dataset_name,
        "Static + temporal",
        data,
        (
            static_numeric
            + temporal_numeric
        ),
        (
            static_categorical
            + temporal_categorical
        ),
        outer_folds
    )

    all_results.extend(
        [
            static_results,
            enhanced_results
        ]
    )

    all_oof.extend(
        [
            static_oof,
            enhanced_oof
        ]
    )


fold_results = pd.concat(
    all_results,
    ignore_index=True
)

oof = pd.concat(
    all_oof,
    ignore_index=True
)

fold_results.to_csv(
    OUTPUT_DIR
    / "temporal_incremental_fold_results.csv",
    index=False
)

oof.to_csv(
    OUTPUT_DIR
    / "temporal_incremental_oof_predictions.csv",
    index=False
)

print(
    "Completed fold-result rows:",
    len(
        fold_results
    )
)



 rainfall | Static only | Logistic Regression

 rainfall | Static only | Random Forest

 rainfall | Static only | XGBoost

 rainfall | Static + temporal | Logistic Regression

 rainfall | Static + temporal | Random Forest

 rainfall | Static + temporal | XGBoost

 rainfall_soilmoisture | Static only | Logistic Regression

 rainfall_soilmoisture | Static only | Random Forest

 rainfall_soilmoisture | Static only | XGBoost

 rainfall_soilmoisture | Static + temporal | Logistic Regression

 rainfall_soilmoisture | Static + temporal | Random Forest

 rainfall_soilmoisture | Static + temporal | XGBoost

 landcover | Static only | Logistic Regression

 landcover | Static only | Random Forest

 landcover | Static only | XGBoost

 landcover | Static + temporal | Logistic Regression

 landcover | Static + temporal | Random Forest

 landcover | Static + temporal | XGBoost

 full_enhanced | Static only | Logistic Regression

 full_enhanced | Static only | Random Forest

 full_enhanced | Static o

# PART F — SUMMARY AND INCREMENTAL VALUE

## 10. Mean test metrics by experiment / algorithm / variant


In [13]:
summary_rows = []

for (
    dataset_name,
    variant,
    model_name
), group in fold_results.groupby(
    [
        "dataset",
        "variant",
        "model"
    ]
):

    row = {
        "dataset":
            dataset_name,

        "variant":
            variant,

        "model":
            model_name,
    }

    for metric in METRIC_NAMES:

        row[
            f"{metric}_mean"
        ] = group[
            f"test_{metric}"
        ].mean()

        row[
            f"{metric}_std"
        ] = group[
            f"test_{metric}"
        ].std()

    row[
        "roc_auc_gap_mean"
    ] = group[
        "gap_roc_auc"
    ].mean()

    summary_rows.append(
        row
    )


summary = pd.DataFrame(
    summary_rows
)

display(
    summary.sort_values(
        [
            "dataset",
            "model",
            "variant"
        ]
    )
)

summary.to_csv(
    OUTPUT_DIR
    / "temporal_incremental_summary.csv",
    index=False
)


,dataset,variant,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,roc_auc_gap_mean
0,full_enhanced,Static + temporal,Logistic Regression,0.837104,0.041613,0.835346,0.052029,0.735727,0.084611,0.830072,0.143656,0.770590,0.062209,0.925323,0.056611,0.891471,0.064882,0.052222
3,full_enhanced,Static only,Logistic Regression,0.754536,0.101776,0.754126,0.110035,0.619749,0.112317,0.752899,0.196258,0.668319,0.126579,0.789136,0.096366,0.671659,0.118219,0.100048
1,full_enhanced,Static + temporal,Random Forest,0.852402,0.036536,0.844827,0.064768,0.762902,0.050512,0.822101,0.163257,0.781840,0.073913,0.940446,0.032171,0.910528,0.045180,0.053686
4,full_enhanced,Static only,Random Forest,0.677228,0.040366,0.692794,0.050768,0.528206,0.068746,0.739493,0.260833,0.591523,0.072199,0.831127,0.092160,0.700535,0.165299,0.153756
2,full_enhanced,Static + temporal,XGBoost,0.821860,0.042050,0.824366,0.080392,0.704992,0.059897,0.831884,0.215996,0.745236,0.102680,0.933252,0.032219,0.899870,0.047515,0.056974
5,full_enhanced,Static only,XGBoost,0.709300,0.101856,0.678754,0.121981,0.564025,0.176886,0.587118,0.251927,0.559811,0.188840,0.802800,0.098590,0.676196,0.183747,0.149851
6,landcover,Static + temporal,Logistic Regression,0.816092,0.075600,0.825115,0.087945,0.678254,0.090542,0.852184,0.129731,0.754584,0.103148,0.889587,0.065630,0.855104,0.064352,0.044138
9,landcover,Static only,Logistic Regression,0.786130,0.053322,0.777126,0.074915,0.654206,0.055894,0.750115,0.146272,0.696098,0.090143,0.815280,0.081069,0.714909,0.101702,0.063166
7,landcover,Static + temporal,Random Forest,0.799693,0.064374,0.804253,0.057376,0.674674,0.098728,0.817931,0.104811,0.733188,0.071997,0.905839,0.056585,0.869920,0.057294,0.068062
10,landcover,Static only,Random Forest,0.747739,0.064774,0.766954,0.048573,0.598147,0.078446,0.824598,0.091938,0.687582,0.054709,0.830586,0.069717,0.713335,0.130532,0.125951


## 11. Calculate improvement caused by temporal predictors

Positive values mean the temporal feature set improved held-out performance on the same sample rows.


In [14]:
static_summary = (
    summary.loc[
        summary[
            "variant"
        ].eq(
            "Static only"
        )
    ]
    .drop(
        columns="variant"
    )
)


temporal_summary = (
    summary.loc[
        summary[
            "variant"
        ].eq(
            "Static + temporal"
        )
    ]
    .drop(
        columns="variant"
    )
)


comparison = static_summary.merge(
    temporal_summary,
    on=[
        "dataset",
        "model"
    ],
    suffixes=(
        "_static",
        "_temporal"
    )
)


for metric in [
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "recall",
    "f1",
    "accuracy",
]:

    comparison[
        f"change_{metric}"
    ] = (
        comparison[
            f"{metric}_mean_temporal"
        ]
        -
        comparison[
            f"{metric}_mean_static"
        ]
    )


comparison = comparison.sort_values(
    [
        "dataset",
        "change_roc_auc"
    ],
    ascending=[
        True,
        False
    ]
)


display(
    comparison[
        [
            "dataset",
            "model",
            "roc_auc_mean_static",
            "roc_auc_mean_temporal",
            "change_roc_auc",
            "pr_auc_mean_static",
            "pr_auc_mean_temporal",
            "change_pr_auc",
            "balanced_accuracy_mean_static",
            "balanced_accuracy_mean_temporal",
            "change_balanced_accuracy",
            "recall_mean_static",
            "recall_mean_temporal",
            "change_recall",
            "f1_mean_static",
            "f1_mean_temporal",
            "change_f1",
        ]
    ]
)


comparison.to_csv(
    OUTPUT_DIR
    / "temporal_incremental_comparison.csv",
    index=False
)


,dataset,model,roc_auc_mean_static,roc_auc_mean_temporal,change_roc_auc,pr_auc_mean_static,pr_auc_mean_temporal,change_pr_auc,balanced_accuracy_mean_static,balanced_accuracy_mean_temporal,change_balanced_accuracy,recall_mean_static,recall_mean_temporal,change_recall,f1_mean_static,f1_mean_temporal,change_f1
0,full_enhanced,Logistic Regression,0.789136,0.925323,0.136187,0.671659,0.891471,0.219812,0.754126,0.835346,0.081220,0.752899,0.830072,0.077174,0.668319,0.770590,0.102271
2,full_enhanced,XGBoost,0.802800,0.933252,0.130452,0.676196,0.899870,0.223674,0.678754,0.824366,0.145612,0.587118,0.831884,0.244767,0.559811,0.745236,0.185426
1,full_enhanced,Random Forest,0.831127,0.940446,0.109319,0.700535,0.910528,0.209993,0.692794,0.844827,0.152033,0.739493,0.822101,0.082609,0.591523,0.781840,0.190316
5,landcover,XGBoost,0.799297,0.878088,0.078791,0.700273,0.839576,0.139302,0.693621,0.782184,0.088563,0.680690,0.688966,0.008276,0.588889,0.711061,0.122171
4,landcover,Random Forest,0.830586,0.905839,0.075253,0.713335,0.869920,0.156585,0.766954,0.804253,0.037299,0.824598,0.817931,-0.006667,0.687582,0.733188,0.045606
3,landcover,Logistic Regression,0.815280,0.889587,0.074307,0.714909,0.855104,0.140195,0.777126,0.825115,0.047989,0.750115,0.852184,0.102069,0.696098,0.754584,0.058487
6,rainfall,Logistic Regression,0.787472,0.823247,0.035775,0.664350,0.723324,0.058974,0.778094,0.787269,0.009175,0.725178,0.798862,0.073684,0.696595,0.712504,0.015909
8,rainfall,XGBoost,0.803944,0.838354,0.034411,0.683026,0.727866,0.044840,0.755654,0.753129,-0.002525,0.720199,0.767710,0.047511,0.671631,0.667176,-0.004454
7,rainfall,Random Forest,0.849434,0.875888,0.026454,0.726962,0.759657,0.032695,0.751209,0.805548,0.054339,0.773257,0.883926,0.110669,0.662762,0.727400,0.064638
9,rainfall_soilmoisture,Logistic Regression,0.804132,0.820970,0.016839,0.647309,0.703011,0.055702,0.755948,0.746220,-0.009728,0.713911,0.706855,-0.007056,0.656398,0.633983,-0.022416


# PART G — OVERALL OOF RESULTS

## 12. Overall held-out performance


In [15]:
overall_rows = []
confusion_rows = []

for (
    dataset_name,
    variant,
    model_name
), group in oof.groupby(
    [
        "dataset",
        "variant",
        "model"
    ]
):

    group = group.sort_values(
        "row_index"
    )

    y_true = group[
        "target"
    ].to_numpy()

    prob = group[
        "probability"
    ].to_numpy()

    pred = (
        prob
        >= group[
            "threshold"
        ].to_numpy()
    ).astype(int)

    overall_rows.append({
        "dataset":
            dataset_name,

        "variant":
            variant,

        "model":
            model_name,

        "accuracy":
            accuracy_score(
                y_true,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                pred
            ),

        "precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                prob
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                prob
            ),
    })

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred
    ).ravel()

    confusion_rows.append({
        "dataset":
            dataset_name,

        "variant":
            variant,

        "model":
            model_name,

        "true_negative":
            int(tn),

        "false_positive":
            int(fp),

        "false_negative":
            int(fn),

        "true_positive":
            int(tp),
    })


overall = pd.DataFrame(
    overall_rows
)

confusion = pd.DataFrame(
    confusion_rows
)


display(
    overall.sort_values(
        [
            "dataset",
            "roc_auc"
        ],
        ascending=[
            True,
            False
        ]
    )
)


overall.to_csv(
    OUTPUT_DIR
    / "temporal_incremental_overall_metrics.csv",
    index=False
)


confusion.to_csv(
    OUTPUT_DIR
    / "temporal_incremental_confusion_matrices.csv",
    index=False
)


,dataset,variant,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc
1,full_enhanced,Static + temporal,Random Forest,0.853994,0.847107,0.757576,0.826446,0.790514,0.935763,0.895780
2,full_enhanced,Static + temporal,XGBoost,0.823691,0.826446,0.696552,0.834711,0.759398,0.923332,0.891741
0,full_enhanced,Static + temporal,Logistic Regression,0.837466,0.836777,0.721429,0.834711,0.773946,0.922410,0.884520
4,full_enhanced,Static only,Random Forest,0.677686,0.694215,0.511364,0.743802,0.606061,0.828871,0.714383
3,full_enhanced,Static only,Logistic Regression,0.757576,0.758264,0.609272,0.760331,0.676471,0.790212,0.674472
5,full_enhanced,Static only,XGBoost,0.710744,0.679752,0.563492,0.586777,0.574899,0.783297,0.645378
7,landcover,Static + temporal,Random Forest,0.799550,0.804054,0.661202,0.817568,0.731118,0.900772,0.861020
6,landcover,Static + temporal,Logistic Regression,0.815315,0.824324,0.677419,0.851351,0.754491,0.880615,0.845309
8,landcover,Static + temporal,XGBoost,0.813063,0.782095,0.733813,0.689189,0.710801,0.861692,0.820343
10,landcover,Static only,Random Forest,0.747748,0.766892,0.586538,0.824324,0.685393,0.828319,0.711145


# PART H — DECISION TABLE

## 13. Identify the strongest temporal experiment

The primary question is not:

> Which dataset gives the single highest score?

The important question is:

> On the **same rows**, did the temporal variables improve the model compared with static predictors alone?

This is what `temporal_incremental_comparison.csv` answers.


In [16]:
decision_cols = [
    "dataset",
    "model",
    "change_roc_auc",
    "change_pr_auc",
    "change_balanced_accuracy",
    "change_recall",
    "change_f1",
]

decision_table = comparison[
    decision_cols
].copy()

decision_table[
    "temporal_improvement_signal"
] = np.select(
    [
        (
            decision_table[
                "change_roc_auc"
            ] > 0
        )
        &
        (
            decision_table[
                "change_pr_auc"
            ] > 0
        ),

        (
            decision_table[
                "change_roc_auc"
            ] > 0
        )
        |
        (
            decision_table[
                "change_pr_auc"
            ] > 0
        ),
    ],
    [
        "Clear improvement",
        "Mixed improvement",
    ],
    default="No clear improvement"
)


display(
    decision_table.sort_values(
        "change_roc_auc",
        ascending=False
    )
)


decision_table.to_csv(
    OUTPUT_DIR
    / "temporal_incremental_decision_table.csv",
    index=False
)


,dataset,model,change_roc_auc,change_pr_auc,change_balanced_accuracy,change_recall,change_f1,temporal_improvement_signal
0,full_enhanced,Logistic Regression,0.136187,0.219812,0.081220,0.077174,0.102271,Clear improvement
2,full_enhanced,XGBoost,0.130452,0.223674,0.145612,0.244767,0.185426,Clear improvement
1,full_enhanced,Random Forest,0.109319,0.209993,0.152033,0.082609,0.190316,Clear improvement
5,landcover,XGBoost,0.078791,0.139302,0.088563,0.008276,0.122171,Clear improvement
4,landcover,Random Forest,0.075253,0.156585,0.037299,-0.006667,0.045606,Clear improvement
3,landcover,Logistic Regression,0.074307,0.140195,0.047989,0.102069,0.058487,Clear improvement
6,rainfall,Logistic Regression,0.035775,0.058974,0.009175,0.073684,0.015909,Clear improvement
8,rainfall,XGBoost,0.034411,0.044840,-0.002525,0.047511,-0.004454,Clear improvement
7,rainfall,Random Forest,0.026454,0.032695,0.054339,0.110669,0.064638,Clear improvement
9,rainfall_soilmoisture,Logistic Regression,0.016839,0.055702,-0.009728,-0.007056,-0.022416,Clear improvement


# What you are training at this stage

## Stage 08

You trained a **static landslide susceptibility model**.

Question:

> Where in Malaysia are environmental conditions more similar to known landslide locations?

The best current static model is Random Forest.

---

## Stage 09

You are training a **semi-dynamic occurrence-condition experiment**.

Question:

> If we already know the static susceptibility conditions, do recent/historical temporal conditions improve our ability to distinguish landslide events from date-matched controls?

Examples:

### Rainfall experiment

```text
Static terrain / soil / geology
        vs
Static terrain / soil / geology
+ antecedent rainfall
```

If the second model performs better on the same cases/controls, rainfall adds useful temporal information.

### Soil-moisture experiment

```text
Static + rainfall
        vs
Static + rainfall + soil moisture
```

This tests whether wet antecedent ground conditions add value.

### Historical land-cover experiment

Tests whether the land-cover state associated with the event year provides information beyond static physical conditions.

---

# What this still does NOT predict

It does not directly predict:

- fatalities;
- building damage;
- road blockage;
- economic loss.

Those require exposure/consequence data.

A road layer can later be overlaid on susceptibility/hazard output to produce:

> Road segments exposed to high landslide susceptibility / semi-dynamic hazard.

That is different from predicting with certainty that a road will be blocked.


# Stage 09 outputs to send

Please send:

```text
temporal_incremental_summary.csv
temporal_incremental_comparison.csv
temporal_incremental_overall_metrics.csv
temporal_incremental_confusion_matrices.csv
temporal_incremental_decision_table.csv
```

The most important file is:

`temporal_incremental_comparison.csv`

because it directly shows whether rainfall, soil moisture, historical land cover and the full temporal feature set genuinely improved prediction on the same sample records.
